In [ ]:
pip install pymupdf pandas tqdm

In [ ]:
import fitz
import pandas as pd
import re
import random
from pathlib import Path

PDF_FOLDER = "educational_pdfs"
OUTPUT_FILE = "educational_dataset_expanded.csv"

LABEL = "educational"

TARGET_SIZE = 3000

BLOCK_WORDS = 80
STEP_WORDS = 30
MIN_WORDS = 45

random.seed(42)

def extract_text_from_pdf(pdf_file):
    doc = fitz.open(pdf_file)
    pages = []

    for page in doc:
        text = page.get_text()
        pages.append(text)

    return "\n".join(pages)

def clean_text(text):
    text = str(text)

    text = text.replace("\r", " ")
    text = text.replace("\n", " ")

    text = re.sub(r"-\s+", "", text)
    text = re.sub(r"\s+", " ", text)

    text = re.sub(r"\?\.", "?", text)
    text = re.sub(r"\!\.", "!", text)

    text = re.sub(r"\.([A-Z])", r". \1", text)
    text = re.sub(r"\?([A-Z])", r"? \1", text)
    text = re.sub(r"\!([A-Z])", r"! \1", text)

    text = re.sub(r"\s+([,.!?;:])", r"\1", text)

    text = text.strip()

    return text

def remove_noise(text):
    text = str(text)

    text = re.sub(r"\bPart\s+[IVXLCDM\d]+\b", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bText\s+\d+\b", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bExercise\s+\d+\b", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bTask\s+\d+\b", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bUnit\s+\d+\b", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bLesson\s+\d+\b", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bChapter\s+\d+\b", " ", text, flags=re.IGNORECASE)

    text = re.sub(r"\bREADER FOR ELEMENTARY STUDENTS\b", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bSome interesting things to read\b", " ", text, flags=re.IGNORECASE)

    text = re.sub(r"\b\d+\b", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

def split_with_overlap(text, block_words=80, step_words=35, min_words=45):
    words = text.split()
    blocks = []

    start = 0

    while start < len(words):
        block_words_list = words[start:start + block_words]

        if len(block_words_list) >= min_words:
            block = " ".join(block_words_list)
            blocks.append(block)

        start += step_words

    return blocks

def is_good_block(text):
    text = str(text)
    words = text.split()

    if len(words) < MIN_WORDS:
        return False

    if re.search(r"[А-Яа-яЁё]", text):
        return False

    letters = re.findall(r"[A-Za-z]", text)
    if len(letters) < 100:
        return False

    english_words = re.findall(r"\b[a-zA-Z]{2,}\b", text)
    if len(english_words) < 30:
        return False

    bad_patterns = [
        "contents",
        "table of contents",
        "reader for elementary students",
        "part i",
        "part ii",
        "part iii",
        "answer the questions",
        "choose the correct answer",
        "fill in the blanks",
        "translate the following",
        "переведите",
        "представленных ниже",
        "ответьте",
        "упражнение",
        "задание",
    ]

    lower = text.lower()

    for pattern in bad_patterns:
        if pattern in lower:
            return False

    return True

pdf_paths = sorted(Path(PDF_FOLDER).glob("*.pdf"))

print("Найдено PDF:", len(pdf_paths))

all_blocks = []

for pdf_path in pdf_paths:
    print("Processing:", pdf_path.name)

    raw_text = extract_text_from_pdf(pdf_path)
    raw_text = remove_noise(raw_text)
    raw_text = clean_text(raw_text)

    blocks = split_with_overlap(
        raw_text,
        block_words=BLOCK_WORDS,
        step_words=STEP_WORDS,
        min_words=MIN_WORDS
    )

    print("Блоков из файла:", len(blocks))

    all_blocks.extend(blocks)

df = pd.DataFrame({
    "text": all_blocks,
    "label": LABEL
})

print("Всего блоков до фильтрации:", len(df))

df = df.dropna()
df["text"] = df["text"].apply(clean_text)

df = df[df["text"].apply(is_good_block)]
df = df.drop_duplicates(subset=["text"])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Всего блоков после фильтрации:", len(df))

if len(df) >= TARGET_SIZE:
    df = df.sample(n=TARGET_SIZE, random_state=42).reset_index(drop=True)
else:
    print("ВНИМАНИЕ: получилось меньше 3000.")
    print("Получилось:", len(df))
    print("Можно добавить ещё PDF или уменьшить STEP_WORDS до 30/25.")

df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

print("Saved:", OUTPUT_FILE)
print(df.shape)
print(df["label"].value_counts())
print(df.head(10))

Найдено PDF: 6
Processing: B1-sample-tests-Reading.pdf
Блоков из файла: 77
Processing: Don-Quixote-By-Miguel-de-Cervantes-book-PDF.pdf
Блоков из файла: 270
Processing: Intermediate-Reading-Passages.pdf
Блоков из файла: 1048
Processing: posobie_4_1_.pdf
Блоков из файла: 1860
Processing: reading-world-a2-b1.pdf
Блоков из файла: 222
Processing: www.ingilizcecin.com-b1-level-english-reading-comprehension-practice-with-answer-key-b1-seviyesi-ingilizce-okuma-parcalari-ve-sorulari-cevaplar-92775.pdf
Блоков из файла: 140
Всего блоков до фильтрации: 3617
Всего блоков после фильтрации: 3341
Saved: educational_dataset_expanded.csv
(3000, 2)
label
educational    3000
Name: count, dtype: int64
                                                text        label
0  agency, and I only went into it because I've a...  educational
1  University is rooted in liberal arts education...  educational
2  one occasion in several houses at a time. The ...  educational
3  of their outlets, and replacing the old nam